# Expanding (running) statistics

An *expanding* statistic grows its window from the first sample to the current
one: every new observation is included and nothing is ever dropped. This
contrasts with a *rolling* window, which keeps only the most recent *n*
samples and slides forward.

Expanding statistics are the natural choice whenever you want a session-to-date
or all-time metric that keeps improving as more data arrives. screamer provides
a family of them: `ExpandingMean`, `ExpandingStd`, `ExpandingVar`,
`ExpandingSkew`, `ExpandingKurt`, `ExpandingSlope`, `ExpandingSum`,
`ExpandingMax`, `ExpandingMin`, and `ExpandingProd`.

Each is a functor: call it with an array to get the full trajectory, or feed
it one value at a time for a live stream. Call `.reset()` to restart from
scratch, useful when a new session or episode begins.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import ExpandingMean, ExpandingStd

SEED = 17
rng = np.random.default_rng(SEED)

# 250-step random walk: cumulative sum of standard-normal increments.
x = np.cumsum(rng.normal(size=250))

In [ ]:
# Compute the expanding mean and expanding standard deviation.
emean = ExpandingMean()(x)
estd  = ExpandingStd()(x)

fig, ax = plt.subplots(figsize=(9, 3.5))

ax.plot(x, lw=0.8, color="0.6", label="random walk")
ax.plot(emean, lw=1.8, color="steelblue", label="expanding mean")

# Shaded +/- 2 standard-deviation band.  The first sample yields NaN for std
# (a single point has no spread), so the band starts at sample 1;
# matplotlib leaves the gap automatically.
ax.fill_between(
    np.arange(len(x)),
    emean - 2 * estd,
    emean + 2 * estd,
    alpha=0.20,
    color="steelblue",
    label="mean +/- 2 std",
)

ax.set_xlabel("sample")
ax.set_title("Expanding mean and +/- 2 sigma band (all-history estimate)")
ax.legend(fontsize=9)
plt.tight_layout()

## Restarting with `.reset()`

Expanding statistics accumulate over *all* history. Call `.reset()` to restart
them, for example at the open of each trading session. Below we reset the
expanding mean and std at `t = 100` and `t = 200`. Each time, the accumulator
starts from scratch, so the +/- 2 sigma band **collapses** (a single fresh
sample has no spread) and then rebuilds as new samples arrive.


In [ ]:
# Reset at t=100 and t=200 by running each segment through a fresh accumulator.
# These are array calls; .reset() clears state between segments (no per-value loop).
resets = [0, 100, 200, len(x)]
em, es = ExpandingMean(), ExpandingStd()
means, stds = [], []
for a, b in zip(resets[:-1], resets[1:]):
    means.append(np.asarray(em(x[a:b])))
    stds.append(np.asarray(es(x[a:b])))
    em.reset(); es.reset()
emean_r = np.concatenate(means)
estd_r  = np.concatenate(stds)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(x, lw=0.8, color="0.6", label="random walk")
ax.plot(emean_r, lw=1.8, color="steelblue", label="expanding mean")
ax.fill_between(
    np.arange(len(x)),
    emean_r - 2 * estd_r,
    emean_r + 2 * estd_r,
    alpha=0.20, color="steelblue", label="mean +/- 2 std",
)
for r in (100, 200):
    ax.axvline(r, color="crimson", lw=1.0, ls="--")
ax.set_xlabel("sample")
ax.set_title("reset() at t=100 and t=200: the band collapses and rebuilds")
ax.legend(fontsize=9)
plt.tight_layout()
